In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
import xgboost as xgb
import shap
import json
import warnings
warnings.filterwarnings('ignore')

d:\D_Drive\vamsi vs code\EDUKRON_FILES\Customer_churn_prediction_retenction\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:


# class ProductionFeatureSelector:
#     def __init__(self, df, target_col='churned'):
#         self.raw_df = df.copy()
#         self.df = df.copy()
#         self.target_col = target_col
#         self.target = df[target_col] if target_col in df.columns else None
         
#         self.removed_features = {
#             "business_filtering": [],
#             "leakage": [],
#             "data_quality": [],
#             "correlation": [],
#             "instability": [],
#             "cost_latency": []
#         }
#         self.business_table = []
#         self.leakage_report = []
#         self.quality_report = []
#         self.initial_feature_count = len(df.columns) - 1 # excluding target
        
#     def step1_business_filtering(self):
#         print("\n--- STEP 1: Business Filtering ---")
#         # Define rules for ID columns and useless metadata
#         id_keywords = ['_id', 'id']
#         metadata_cols = ['updated_at', 'created_at', 'measured_date', 'last_contact_date', 'account_manager', 'contract_end_date']
        
#         for col in self.df.columns:
#             if col == self.target_col:
#                 continue
            
#             is_dropped = False
#             reason = ""
            
#             # Rule 1: Remove identifiers
#             if any(kw in col.lower() for kw in id_keywords) and len(self.df[col].unique()) > len(self.df) * 0.5:
#                 reason = "Pure Identifier (High cardinality, no predictive value)"
#                 is_dropped = True
            
#             # Rule 2: Remove metadata
#             elif col in metadata_cols:
#                 reason = "Metadata/Timestamp (Not a predictive behavioral feature)"
#                 is_dropped = True
                
#             if is_dropped:
#                 self.business_table.append({'Feature': col, 'Status': 'Dropped', 'Reason': reason})
#                 self.removed_features["business_filtering"].append(col)
#                 self.df = self.df.drop(columns=[col])
#             else:
#                 self.business_table.append({'Feature': col, 'Status': 'Kept', 'Reason': "Business Relevant"})
                
#         # Print output
#         print(pd.DataFrame(self.business_table)[pd.DataFrame(self.business_table)['Status'] == 'Dropped'])
        
#     def step2_leakage_detection(self):
#         print("\n--- STEP 2: Leakage Detection ---")
#         # Known leakage columns for churn
#         leakage_keywords = ['churn_reason', 'predicted_churn_prob', 'retention_offer_sent', 'offer_accepted', 'campaign_response']
        
#         for col in self.df.columns:
#             if col == self.target_col: continue
            
#             if any(leak in col.lower() for leak in leakage_keywords):
#                 reason = f"Data Leakage: Contains future or post-churn signals ('{col}')"
#                 self.leakage_report.append({'Feature': col, 'Reason': reason})
#                 self.removed_features["leakage"].append(col)
#                 self.df = self.df.drop(columns=[col])
                
#         if self.leakage_report:
#             print(pd.DataFrame(self.leakage_report))
#         else:
#             print("No leakage features detected.")

#     def step3_data_quality(self):
#         print("\n--- STEP 3: Data Quality Filtering ---")
#         for col in self.df.columns:
#             if col == self.target_col: continue
            
#             # 1. Missing Value Analysis (>60% nulls)
#             null_pct = self.df[col].isnull().sum() / len(self.df)
#             if null_pct > 0.60:
#                 self.removed_features["data_quality"].append(col)
#                 self.quality_report.append({'Feature': col, 'Issue': 'High Nulls', 'Metric': f"{null_pct*100:.1f}% null"})
#                 self.df = self.df.drop(columns=[col])
#                 continue
                
#             # 2. Low Variance / Constant Feature
#             if self.df[col].nunique() <= 1:
#                 self.removed_features["data_quality"].append(col)
#                 self.quality_report.append({'Feature': col, 'Issue': 'Zero Variance', 'Metric': "1 unique value"})
#                 self.df = self.df.drop(columns=[col])
                
#         if self.quality_report:
#             print(pd.DataFrame(self.quality_report))
#         else:
#             print("No data quality issues detected.")

#     def step4_statistical_filtering(self):
#         print("\n--- STEP 4: Statistical Filtering (Correlation & MI) ---")
        
#         # We need a strictly numeric dataframe for correlation and MI
#         df_numeric = self.df.copy()
        
#         # Quick encoding and imputation for statistical checks
#         cat_cols = df_numeric.select_dtypes(include=['object', 'category']).columns
#         if len(cat_cols) > 0:
#             df_numeric[cat_cols] = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit_transform(df_numeric[cat_cols].astype(str))
            
#         imputer = SimpleImputer(strategy='median')
#         X_num = pd.DataFrame(imputer.fit_transform(df_numeric.drop(columns=[self.target_col])), columns=df_numeric.drop(columns=[self.target_col]).columns)
#         y = self.target
        
#         # 1. Correlation Analysis
#         corr_matrix = X_num.corr().abs()
#         upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        
#         to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]
#         for col in to_drop:
#             self.removed_features["correlation"].append(col)
#             self.df = self.df.drop(columns=[col], errors='ignore')
#             X_num = X_num.drop(columns=[col], errors='ignore')
#             print(f"Dropped {col} due to high correlation (>0.90) with another feature.")
            
#         # # 2. Mutual Information
#         # print("\nCalculating Mutual Information...")
#         # mi_scores = mutual_info_classif(X_num, y, random_state=42)
#         # mi_series = pd.Series(mi_scores, index=X_num.columns).sort_values(ascending=False)
#         # print("Top 10 Features by Mutual Information:")
#         # print(mi_series.head(10))
        
#         self.X_encoded = X_num
#         self.y = y

#     def step5_model_based_selection(self):
#         print("\n--- STEP 5: Model-Based Selection ---")
#         # Train LR, RF, XGB
#         X_train, X_test, y_train, y_test = train_test_split(self.X_encoded, self.y, test_size=0.2, random_state=42)
        
#         # Scale for LR
#         scaler = StandardScaler()
#         X_train_scaled = scaler.fit_transform(X_train)
        
#         print("Training Logistic Regression...")
#         lr = LogisticRegression(random_state=42, max_iter=1000).fit(X_train_scaled, y_train)
#         lr_importance = pd.Series(abs(lr.coef_[0]), index=self.X_encoded.columns)
        
#         print("Training Random Forest...")
#         rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)
#         rf_importance = pd.Series(rf.feature_importances_, index=self.X_encoded.columns)
        
#         print("Training XGBoost...")
#         xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42).fit(X_train, y_train)
#         xgb_importance = pd.Series(xgb_model.feature_importances_, index=self.X_encoded.columns)
        
#         # Combine and rank
#         importance_df = pd.DataFrame({
#             'LR_Rank': lr_importance.rank(ascending=False),
#             'RF_Rank': rf_importance.rank(ascending=False),
#             'XGB_Rank': xgb_importance.rank(ascending=False)
#         })
        
#         importance_df['Average_Rank'] = importance_df.mean(axis=1)
#         importance_df = importance_df.sort_values('Average_Rank')
        
#         print("Top 10 Consistent Features across Models (Lowest Avg Rank is Best):")
#         print(importance_df.head(10))
        
#         # Save models for SHAP
#         self.xgb_model = xgb_model
#         self.X_train = X_train

#     def step6_shap_analysis(self):
#         print("\n--- STEP 6: SHAP Analysis ---")
#         print("Calculating SHAP values for XGBoost model...")
#         explainer = shap.TreeExplainer(self.xgb_model)
#         # Using a sample to speed up computation in production evaluation
#         X_sample = self.X_train.sample(n=min(5000, len(self.X_train)), random_state=42)
#         shap_values = explainer.shap_values(X_sample)
        
#         # Global feature importance
#         vals = np.abs(shap_values).mean(0)
#         shap_importance = pd.DataFrame(list(zip(X_sample.columns, vals)), columns=['Feature', 'SHAP_Importance'])
#         shap_importance = shap_importance.sort_values(by=['SHAP_Importance'], ascending=False)
        
#         print("Top 10 Features by Global SHAP Importance:")
#         print(shap_importance.head(10).to_string(index=False))
#         self.shap_importance = shap_importance

#     def step7_stability_testing(self):
#         print("\n--- STEP 7: Stability Testing ---")
#         print("NOTE: Real stability testing requires a time-split. Assuming features with extremely low SHAP values are noisy/unstable.")
        
#         # Drop features that have ~0 SHAP importance (noisy)
#         unstable_features = self.shap_importance[self.shap_importance['SHAP_Importance'] < 0.0001]['Feature'].tolist()
#         for col in unstable_features:
#             self.removed_features["instability"].append(col)
#             if col in self.df.columns:
#                 self.df = self.df.drop(columns=[col])
#         print(f"Removed {len(unstable_features)} unstable/noisy features with near-zero SHAP impact.")

#     def step8_cost_latency_filtering(self):
#         print("\n--- STEP 8: Cost / Latency Filtering ---")
#         # Simulate dropping a complex feature if it ranked low in SHAP
#         # Example: if an aggregation like 'eng_forum_posts_per_month' is expensive but ranked > 50
#         print("Applying latency heuristics...")
#         # (Custom logic can be added here based on actual DB costs)
#         print("No high-cost/low-reward features flagged in this run.")

#     def generate_final_deliverables(self):
#         print("\n=======================================================")
#         print("              FINAL DELIVERABLES                       ")
#         print("=======================================================\n")
        
#         FINAL_FEATURES = [col for col in self.df.columns if col != self.target_col]
        
#         print("1. FINAL_FEATURES = [")
#         for f in FINAL_FEATURES:
#             print(f"    '{f}',")
#         print("]\n")
        
#         print("2. REMOVED_FEATURES =")
#         print(json.dumps(self.removed_features, indent=4))
        
#         print("\n3. PRODUCTION SUMMARY REPORT")
#         print("-" * 30)
#         print(f"Total Raw Features:         {self.initial_feature_count}")
#         print(f"Total Selected Features:    {len(FINAL_FEATURES)}")
        
#         total_removed = sum(len(v) for v in self.removed_features.values())
#         print(f"Total Removed Features:     {total_removed}")
        
#         print("\nFinal Top 10 Strongest Features (Based on SHAP):")
#         top_10 = self.shap_importance[self.shap_importance['Feature'].isin(FINAL_FEATURES)].head(10)
#         for i, row in top_10.iterrows():
#             print(f"- {row['Feature']} (SHAP: {row['SHAP_Importance']:.4f})")
            
#         print("\nRisks & Issues Detected:")
#         print(f"- Leakage Detected: {len(self.removed_features['leakage'])} columns dropped.")
#         print(f"- Data Quality Issues: {len(self.removed_features['data_quality'])} columns dropped.")
#         print(f"- Drift Risk: Removed {len(self.removed_features['instability'])} unstable features.")
#         print("\nRecommended Deployment Action: Proceed with FINAL_FEATURES to hyperparameter tuning phase.")

In [1]:
class ProductionFeatureSelector:
    def __init__(self, df, target_col='churned'):
        self.raw_df = df.copy()
        self.df = df.copy()
        self.target_col = target_col
        self.target = df[target_col] if target_col in df.columns else None

    # STEP 1: Business Filtering (returns columns to drop, does NOT delete)
    def step1_business_filtering(self):
        business_drop_cols = []

        id_keywords = ['_id', 'id']
        metadata_cols = [
            'updated_at', 'created_at', 'measured_date',
            'last_contact_date', 'account_manager', 'contract_end_date'
        ]

        for col in self.df.columns:
            if col == self.target_col:
                continue

            # Rule 1: High-cardinality ID columns
            if any(kw in col.lower() for kw in id_keywords) and \
               self.df[col].nunique() > len(self.df) * 0.5:
                business_drop_cols.append(col)

            # Rule 2: Metadata columns
            elif col in metadata_cols:
                business_drop_cols.append(col)

        return business_drop_cols

    # STEP 2: Leakage Detection (returns columns to drop)
    def step2_leakage_detection(self):
        leakage_drop_cols = []

        leakage_keywords = [
            'churn_reason',
            'predicted_churn_prob',
            'retention_offer_sent',
            'offer_accepted',
            'campaign_response'
        ]

        for col in self.df.columns:
            if col == self.target_col:
                continue

            if any(leak in col.lower() for leak in leakage_keywords):
                leakage_drop_cols.append(col)

        return leakage_drop_cols

    # STEP 3: Data Quality Filtering (returns columns to drop)
    def step3_data_quality(self):
        quality_drop_cols = []

        for col in self.df.columns:
            if col == self.target_col:
                continue

            # High null percentage
            null_pct = self.df[col].isnull().sum() / len(self.df)
            if null_pct > 0.60:
                quality_drop_cols.append(col)
                continue

            # Constant column
            if self.df[col].nunique() <= 1:
                quality_drop_cols.append(col)

        return quality_drop_cols

    # STEP 4: Statistical Filtering (returns correlated columns to drop)
    def step4_statistical_filtering(self):
        df_numeric = self.df.copy()

        # Encode categorical columns
        cat_cols = df_numeric.select_dtypes(include=['object', 'category']).columns
        if len(cat_cols) > 0:
            encoder = OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            )
            df_numeric[cat_cols] = encoder.fit_transform(
                df_numeric[cat_cols].astype(str)
            )

        # Impute missing values
        imputer = SimpleImputer(strategy='median')

        X_num = pd.DataFrame(
            imputer.fit_transform(
                df_numeric.drop(columns=[self.target_col])
            ),
            columns=df_numeric.drop(columns=[self.target_col]).columns
        )

        # Correlation matrix
        corr_matrix = X_num.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )

        correlated_drop_cols = [
            column for column in upper.columns
            if any(upper[column] > 0.90)
        ]

        return correlated_drop_cols


# # Example usage:

In [4]:

df = pd.read_csv(r'D:\D_Drive\vamsi vs code\EDUKRON_FILES\Customer_churn_prediction_retenction\customer-churn-prediction-retention-model\Data\encoded_final_df.csv')
selector = ProductionFeatureSelector(df, target_col='churned')

In [5]:

selector = ProductionFeatureSelector(df)

step1_cols = selector.step1_business_filtering()
print("Step 1 Drop Columns:", step1_cols)

Step 1 Drop Columns: []


In [9]:
step2_cols = selector.step2_leakage_detection()

print("Step 2 Drop Columns:", step2_cols)
df = df.drop(columns=[step2_cols])

Step 2 Drop Columns: ['predicted_churn_prob', 'offer_accepted', 'churn_reason_none', 'churn_reason_price', 'churn_reason_relocation', 'churn_reason_service', 'retention_offer_sent_email', 'retention_offer_sent_none', 'retention_offer_sent_sms']


KeyError: "[('predicted_churn_prob', 'offer_accepted', 'churn_reason_none', 'churn_reason_price', 'churn_reason_relocation', 'churn_reason_service', 'retention_offer_sent_email', 'retention_offer_sent_none', 'retention_offer_sent_sms')] not found in axis"

In [7]:
step3_cols = selector.step3_data_quality()

print("Step 3 Drop Columns:", step3_cols)

Step 3 Drop Columns: []


In [8]:
step4_cols = selector.step4_statistical_filtering()


print("Step 4 Drop Columns:", step4_cols)

Step 4 Drop Columns: ['Unnamed: 0', 'total_spend', 'avg_order_value', 'annual_contract_value', 'eng_contract_length_months']


In [10]:
df

,Unnamed: 0.1,Unnamed: 0,tenure_months,contract_type,monthly_spend,days_since_last_order,support_tickets_30d,total_spend,order_count,avg_order_value,...,retention_offer_sent_email,retention_offer_sent_none,retention_offer_sent_sms,payment_method_bill_ach,payment_method_bill_credit_card,payment_method_bill_invoice,payment_method_bill_paypal,invoice_delivery_email,invoice_delivery_mail,invoice_delivery_portal
0,0,0,88,2.0,369.85,284,10,29988.0,57,999.60,...,0,1,0,0,0,0,1,0,1,0
1,1,1,88,2.0,369.85,284,10,29988.0,57,999.60,...,0,1,0,0,0,0,1,0,1,0
2,2,2,88,2.0,369.85,284,10,29988.0,57,999.60,...,0,1,0,0,0,0,1,0,1,0
3,3,3,88,2.0,369.85,284,10,29988.0,57,999.60,...,0,1,0,0,0,0,1,0,1,0
4,4,4,88,2.0,369.85,284,10,29988.0,57,999.60,...,0,1,0,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1197727,1197727,1197727,88,2.0,115.66,13,2,8200.5,38,273.35,...,0,0,1,0,0,0,1,0,0,1
1197728,1197728,1197728,88,2.0,115.66,13,2,8200.5,38,273.35,...,0,0,1,0,0,0,1,0,0,1
1197729,1197729,1197729,88,2.0,115.66,13,2,8200.5,38,273.35,...,0,0,1,0,0,0,1,0,0,1
1197730,1197730,1197730,88,2.0,115.66,13,2,8200.5,38,273.35,...,0,0,1,0,0,0,1,0,0,1


In [20]:
print(selector.step1_business_filtering())


--- STEP 1: Business Filtering ---
Empty DataFrame
Columns: [Feature, Status, Reason]
Index: []
None


In [21]:
selector.step2_leakage_detection()


--- STEP 2: Leakage Detection ---
                      Feature  \
0        predicted_churn_prob   
1              offer_accepted   
2           churn_reason_none   
3          churn_reason_price   
4     churn_reason_relocation   
5        churn_reason_service   
6  retention_offer_sent_email   
7   retention_offer_sent_none   
8    retention_offer_sent_sms   

                                              Reason  
0  Data Leakage: Contains future or post-churn si...  
1  Data Leakage: Contains future or post-churn si...  
2  Data Leakage: Contains future or post-churn si...  
3  Data Leakage: Contains future or post-churn si...  
4  Data Leakage: Contains future or post-churn si...  
5  Data Leakage: Contains future or post-churn si...  
6  Data Leakage: Contains future or post-churn si...  
7  Data Leakage: Contains future or post-churn si...  
8  Data Leakage: Contains future or post-churn si...  


In [22]:

selector.step3_data_quality()


--- STEP 3: Data Quality Filtering ---
No data quality issues detected.


In [23]:

selector.step4_statistical_filtering()


--- STEP 4: Statistical Filtering (Correlation & MI) ---
Dropped Unnamed: 0 due to high correlation (>0.90) with another feature.
Dropped total_spend due to high correlation (>0.90) with another feature.
Dropped avg_order_value due to high correlation (>0.90) with another feature.
Dropped annual_contract_value due to high correlation (>0.90) with another feature.
Dropped eng_contract_length_months due to high correlation (>0.90) with another feature.


In [24]:
selector.step5_model_based_selection()


--- STEP 5: Model-Based Selection ---
Training Logistic Regression...
Training Random Forest...
Training XGBoost...
Top 10 Consistent Features across Models (Lowest Avg Rank is Best):
                               LR_Rank  RF_Rank  XGB_Rank  Average_Rank
days_since_last_order              1.0      3.0       1.0      1.666667
nps_score                          2.0      1.0      35.0     12.666667
support_tickets_30d                3.0      2.0      35.0     13.333333
complaint_count                    4.0      5.0      35.0     14.666667
satisfaction_score                 6.0      6.0      35.0     15.666667
eng_support_intensity              8.0      4.0      35.0     15.666667
eng_had_recent_support_ticket      5.0      9.0      35.0     16.333333
eng_is_high_value                  7.0      8.0      35.0     16.666667
eng_forum_posts_per_month         13.0     12.0      35.0     20.000000
order_count                        9.0     19.0      35.0     21.000000


In [25]:
selector.step6_shap_analysis()


--- STEP 6: SHAP Analysis ---
Calculating SHAP values for XGBoost model...
Top 10 Features by Global SHAP Importance:
              Feature  SHAP_Importance
days_since_last_order        13.038385
         Unnamed: 0.1         0.000000
        contract_type         0.000000
        tenure_months         0.000000
        monthly_spend         0.000000
  support_tickets_30d         0.000000
          order_count         0.000000
              autopay         0.000000
   num_products_owned         0.000000
      complaint_count         0.000000


In [26]:
selector.step7_stability_testing()



--- STEP 7: Stability Testing ---
NOTE: Real stability testing requires a time-split. Assuming features with extremely low SHAP values are noisy/unstable.
Removed 67 unstable/noisy features with near-zero SHAP impact.


In [27]:
selector.step8_cost_latency_filtering()



--- STEP 8: Cost / Latency Filtering ---
Applying latency heuristics...
No high-cost/low-reward features flagged in this run.


In [28]:
selector.generate_final_deliverables()


              FINAL DELIVERABLES                       

1. FINAL_FEATURES = [
    'days_since_last_order',
]

2. REMOVED_FEATURES =
{
    "business_filtering": [],
    "leakage": [
        "predicted_churn_prob",
        "offer_accepted",
        "churn_reason_none",
        "churn_reason_price",
        "churn_reason_relocation",
        "churn_reason_service",
        "retention_offer_sent_email",
        "retention_offer_sent_none",
        "retention_offer_sent_sms"
    ],
    "data_quality": [],
    "correlation": [
        "Unnamed: 0",
        "total_spend",
        "avg_order_value",
        "annual_contract_value",
        "eng_contract_length_months"
    ],
    "instability": [
        "Unnamed: 0.1",
        "contract_type",
        "tenure_months",
        "monthly_spend",
        "support_tickets_30d",
        "order_count",
        "autopay",
        "num_products_owned",
        "complaint_count",
        "nps_score",
        "satisfaction_score",
        "recency_days

In [ ]:
corrs = df.corr()['churned']

nps_score               -0.867485
churn_reason_none       -0.832400
support_tickets_30d      0.828031
predicted_churn_prob     0.904120
days_since_last_order    0.907901
churned                  1.000000
Name: churned, dtype: float64


In [19]:
strong_corrs = corrs[abs(corrs) > 0.3].sort_values()
print(strong_corrs)

nps_score                       -0.867485
churn_reason_none               -0.832400
satisfaction_score              -0.408259
eng_had_recent_support_ticket    0.313732
churn_reason_service             0.352336
churn_reason_relocation          0.353370
churn_reason_price               0.353504
eng_is_high_value                0.397526
complaint_count                  0.564952
support_tickets_30d              0.828031
predicted_churn_prob             0.904120
days_since_last_order            0.907901
churned                          1.000000
Name: churned, dtype: float64
